# 16 — E4: detector dedicato alla palla

Il detector multi-classe (E1) è addestrato a 640 px su 464 immagini in cui la palla è una delle 10 classi: in un'inquadratura larga la palla diventa di 5–6 px e viene trovata male (notebook 14 e 15). Qui si addestra un modello **con la sola classe palla**, **a 1280 px**, con in più **frame delle tue clip NBA** annotati a mano, e lo si confronta con il detector attuale.

Il notebook ha **due fasi**, separate da un lavoro manuale:

| Fase | Sezioni | Cosa succede |
|---|---|---|
| **1. Preparazione** | 1–2 | Estrae ~200 frame dalle tue clip NBA e li **pre-annota** con il detector attuale → zip da caricare su Roboflow |
| ✋ **Lavoro manuale** | — | Su Roboflow correggi le box della palla (1–2 ore), poi esporti il dataset |
| **2. Addestramento e valutazione** | 3–7 | Unisce i dataset, addestra, confronta vecchio e nuovo detector sui **tuoi frame tenuti da parte**, misura la percentuale di frame con palla su una clip intera e salva `models/ball_detector.pt` per il notebook 15 |

Runtime: **Colab GPU** (L4 consigliata per l'addestramento a 1280; T4 funziona, più lenta). Serve la API key Roboflow.

In [ ]:
%pip install -q ultralytics roboflow
# dopo il pip: Runtime -> Riavvia sessione, poi esegui dalla cella successiva

In [ ]:
import sys
if 'triton' not in sys.modules: sys.modules['triton'] = None

# ==== SETUP COMUNE (uguale in tutti i notebook) ====
import sys, os
from pathlib import Path
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/Tesi_Basket')   # progetto su Drive
    FAST = Path('/content/fast')                                 # disco locale Colab (veloce)
else:
    ROOT = Path(r'F:\Tesi Basket')                               # progetto in locale
    FAST = ROOT / '_cache'
FAST.mkdir(parents=True, exist_ok=True)
assert ROOT.is_dir(), f'Cartella del progetto non trovata: {ROOT}'
print('Progetto:', ROOT)

import json, re, shutil, random, zipfile, yaml, time
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from ultralytics import YOLO

E4 = ROOT / 'experiments' / 'E4_ball'; E4.mkdir(parents=True, exist_ok=True)
OWN_DIR = ROOT / 'datasets' / 'ball_own'                    # frame delle tue clip (su Drive)
OLD_WEIGHTS = ROOT / 'models' / 'yolo11m_ft.pt'             # detector attuale (E1)
def api_key():
    try:
        from google.colab import userdata; k = userdata.get('ROBOFLOW_API_KEY')
        if k: return k
    except Exception: pass
    return os.environ.get('ROBOFLOW_API_KEY') or __import__('getpass').getpass('API key Roboflow: ').strip()

## 1. Configurazione

`CLIPS`: le clip NBA da cui estrarre i frame (in `videos/`). Più partite diverse = modello più robusto. I frame vengono presi distanziati nel tempo (niente quasi-duplicati) e il **30% finale di ogni clip** viene tenuto da parte come **test**: il modello non lo vede mai in addestramento.

In [ ]:
CLIPS = sorted((ROOT / 'videos').glob('nba_*.mp4'))         # oppure una lista: [ROOT/'videos'/'nba_clip_01.mp4', ...]
FRAMES_TOTAL = 200          # frame da estrarre in totale (divisi tra le clip)
MIN_GAP_S = 0.5             # distanza minima tra due frame estratti
TEST_FRACTION = 0.30        # ultima parte di ogni clip = test
PRE_CONF = 0.15             # soglia della pre-annotazione (meglio qualche box in più da cancellare che palle mancanti)
IMG_W = 1280                # i frame vengono salvati a questa larghezza

# Dataset di partenza (lo stesso di E1): ne usiamo solo le box della palla
RF_WORKSPACE, RF_PROJECT, RF_VERSION = 'roboflow-jvuqo', 'basketball-player-detection-3-ycjdo', 18
# Dataset pubblici aggiuntivi (facoltativi): (workspace, progetto, versione, [nomi della classe palla])
EXTRA_DATASETS = []          # es. [('workspace', 'progetto', 3, ['ball', 'basketball'])]

# Il tuo dataset annotato (fase 2): metti lo zip esportato da Roboflow qui, oppure compila workspace/progetto/versione
OWN_ZIP = ROOT / 'datasets' / 'ball_own_annotato.zip'
OWN_RF = None                # es. ('tuo-workspace', 'palla-nba', 1)

# Addestramento
BASE_MODEL = 'yolo11s.pt'; TRAIN_IMGSZ = 1280; EPOCHS = 100; PATIENCE = 25; SEED = 0
print('Clip trovate:', [c.name for c in CLIPS])

## 2. FASE 1 — Estrazione e pre-annotazione dei frame

In [ ]:
assert CLIPS, 'Nessuna clip trovata: controlla CLIPS'
old = YOLO(str(OLD_WEIGHTS)); OLD_BALL = [i for i, n in old.names.items() if n == 'ball']
(OWN_DIR / 'images').mkdir(parents=True, exist_ok=True); (OWN_DIR / 'labels').mkdir(parents=True, exist_ok=True)
per_clip = max(1, FRAMES_TOTAL // len(CLIPS)); manifest = []
for clip in CLIPS:
    cap = cv2.VideoCapture(str(clip)); fps = cap.get(cv2.CAP_PROP_FPS) or 30; n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    step = max(int(MIN_GAP_S * fps), n // per_clip)
    idx = list(range(0, n, step))[:per_clip]
    for fi in tqdm(idx, desc=clip.stem):
        cap.set(cv2.CAP_PROP_POS_FRAMES, fi); ok, f = cap.read()
        if not ok: continue
        if f.shape[1] != IMG_W: f = cv2.resize(f, (IMG_W, int(round(f.shape[0] * IMG_W / f.shape[1]))), interpolation=cv2.INTER_AREA)
        name = f'{clip.stem}_f{fi:06d}'
        cv2.imwrite(str(OWN_DIR / 'images' / f'{name}.jpg'), f, [cv2.IMWRITE_JPEG_QUALITY, 95])
        r = old.predict(f, imgsz=1280, conf=PRE_CONF, classes=OLD_BALL, verbose=False)[0]
        h, w = f.shape[:2]; lines = []
        for x1, y1, x2, y2 in r.boxes.xyxy.cpu().numpy():
            lines.append(f'0 {(x1 + x2) / 2 / w:.6f} {(y1 + y2) / 2 / h:.6f} {(x2 - x1) / w:.6f} {(y2 - y1) / h:.6f}')
        (OWN_DIR / 'labels' / f'{name}.txt').write_text('\n'.join(lines))
        manifest.append({'file': name, 'clip': clip.stem, 'frame': fi, 'n_frames_clip': n, 'pre_box': len(lines)})
    cap.release()
man = pd.DataFrame(manifest); man.to_csv(OWN_DIR / 'manifest.csv', index=False)
(OWN_DIR / 'data.yaml').write_text(yaml.safe_dump({'names': ['ball'], 'nc': 1}))
zp = ROOT / 'datasets' / 'ball_own_da_annotare.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in (OWN_DIR / 'images').glob('*.jpg'): z.write(p, f'images/{p.name}')
    for p in (OWN_DIR / 'labels').glob('*.txt'): z.write(p, f'labels/{p.name}')
    z.write(OWN_DIR / 'data.yaml', 'data.yaml')
print(f'{len(man)} frame estratti; con almeno una palla pre-annotata: {(man.pre_box > 0).mean():.0%}')
print('Zip da caricare su Roboflow:', zp)

### ✋ Lavoro manuale su Roboflow (1–2 ore)

1. Scarica `datasets/ball_own_da_annotare.zip` da Drive.
2. Su [app.roboflow.com](https://app.roboflow.com): **Create New Project** → tipo **Object Detection**, nome per es. `palla-nba`, classe `ball`.
3. **Upload** → trascina lo zip: Roboflow legge immagini e box pre-annotate.
4. **Annotate**: per ogni immagine
   - **cancella** le box sbagliate (scarpe, teste, luci, loghi…);
   - **aggiungi** la palla dove manca (anche se piccola o parzialmente coperta: box stretta intorno alla parte visibile);
   - se la palla **non si vede** (nascosta dalle mani, fuori campo) lascia l'immagine **senza box**: serve come esempio negativo.
   - Tieni **una sola classe**: `ball`.
5. **Generate → New Version**: in *Preprocessing* **togli il Resize** (le immagini devono restare a 1280 px), **nessuna Augmentation**. Lo split train/valid/test di Roboflow non conta: lo rifà il notebook.
6. Esporta: **Export Dataset → formato YOLOv11 → Download zip** e salvalo su Drive come `datasets/ball_own_annotato.zip` (oppure scrivi workspace/progetto/versione in `OWN_RF`).

Poi esegui la **FASE 2** (sezioni 3–7).

## 3. FASE 2 — Costruzione del dataset della palla

In [ ]:
from roboflow import Roboflow
DS = FAST / 'ball_ds'
if DS.exists(): shutil.rmtree(DS)
for s in ['train', 'val', 'test_own', 'test_v18']:
    (DS / s / 'images').mkdir(parents=True); (DS / s / 'labels').mkdir(parents=True)

def yolo_export_dirs(root):
    # tutte le coppie (immagine, label) di un export YOLO, qualunque sia lo split
    pairs = []
    for img in root.rglob('*.jpg'):
        lab = img.parent.parent / 'labels' / (img.stem + '.txt')
        pairs.append((img, lab if lab.is_file() else None))
    return pairs

def names_of(root):
    y = next(root.rglob('data.yaml'), None)
    if y is None: return ['ball']
    n = yaml.safe_load(y.read_text())['names']
    return n if isinstance(n, list) else [n[k] for k in sorted(n)]

def copy_pair(img, lab, split, keep_ids, prefix):
    name = f'{prefix}_{img.stem}'
    shutil.copy(img, DS / split / 'images' / f'{name}.jpg')
    lines = []
    if lab is not None:
        for l in lab.read_text().splitlines():
            p = l.split()
            if len(p) >= 5 and int(p[0]) in keep_ids: lines.append('0 ' + ' '.join(p[1:5]))
    (DS / split / 'labels' / f'{name}.txt').write_text('\n'.join(lines))
    return len(lines)

stats = []
# --- a) i tuoi frame annotati: split temporale per clip (ultimo 30% = test) ---
own_root = FAST / 'ball_own_annotato'
if own_root.exists(): shutil.rmtree(own_root)
if OWN_ZIP.is_file():
    zipfile.ZipFile(OWN_ZIP).extractall(own_root)
elif OWN_RF:
    Roboflow(api_key=api_key()).workspace(OWN_RF[0]).project(OWN_RF[1]).version(OWN_RF[2]).download('yolov11', location=str(own_root))
else:
    raise FileNotFoundError(f'Manca il dataset annotato: metti lo zip in {OWN_ZIP} oppure compila OWN_RF')
man = pd.read_csv(OWN_DIR / 'manifest.csv'); last = man.groupby('clip').frame.max().to_dict()
own_names = names_of(own_root); own_ids = {i for i, n in enumerate(own_names) if 'ball' in n.lower() and 'basket' not in n.lower()}
widths = []
for img, lab in yolo_export_dirs(own_root):
    m = re.search(r'(.+)_f(\d{6})', img.stem)
    if not m: continue
    clip, fi = m.group(1), int(m.group(2))
    split = 'test_own' if fi >= (1 - TEST_FRACTION) * last.get(clip, fi + 1) else 'train'
    nb_ = copy_pair(img, lab, split, own_ids, 'own'); widths.append(cv2.imread(str(img)).shape[1])
    stats.append({'fonte': 'clip NBA (tue)', 'split': split, 'palle': nb_})
if widths and np.median(widths) < 1000:
    print(f'ATTENZIONE: le immagini esportate sono larghe {int(np.median(widths))} px: su Roboflow hai lasciato il Resize. Rifai la versione senza Resize.')

# --- b) dataset v18: solo le box della palla; test v18 tenuto per controllo ---
v18 = FAST / 'datasets' / f'{RF_PROJECT}-{RF_VERSION}'
if not v18.exists():
    Roboflow(api_key=api_key()).workspace(RF_WORKSPACE).project(RF_PROJECT).version(RF_VERSION).download('yolov11', location=str(v18))
v18_ids = {i for i, n in enumerate(names_of(v18)) if n == 'ball'}
for split_src, split_dst in [('train', 'train'), ('valid', 'train'), ('test', 'test_v18')]:
    for img in (v18 / split_src / 'images').glob('*.jpg'):
        lab = v18 / split_src / 'labels' / (img.stem + '.txt')
        stats.append({'fonte': 'v18', 'split': split_dst, 'palle': copy_pair(img, lab if lab.is_file() else None, split_dst, v18_ids, 'v18')})

# --- c) dataset pubblici aggiuntivi (solo train) ---
for ws, pj, ver, bnames in EXTRA_DATASETS:
    root = FAST / 'datasets' / f'{pj}-{ver}'
    if not root.exists(): Roboflow(api_key=api_key()).workspace(ws).project(pj).version(ver).download('yolov11', location=str(root))
    ids = {i for i, n in enumerate(names_of(root)) if n.lower() in [b.lower() for b in bnames]}
    for img, lab in yolo_export_dirs(root):
        stats.append({'fonte': pj, 'split': 'train', 'palle': copy_pair(img, lab, 'train', ids, pj[:12])})

# --- validation: 15% del train, scelto a caso ma con seme fisso ---
random.seed(SEED); tr = sorted((DS / 'train' / 'images').glob('*.jpg')); random.shuffle(tr)
for img in tr[:int(.15 * len(tr))]:
    shutil.move(str(img), DS / 'val' / 'images' / img.name)
    shutil.move(str(DS / 'train' / 'labels' / (img.stem + '.txt')), DS / 'val' / 'labels' / (img.stem + '.txt'))
st = pd.DataFrame(stats).groupby(['fonte', 'split']).agg(immagini=('palle', 'size'), palle=('palle', 'sum'))
display(st); st.to_csv(E4 / 'dataset_palla.csv')
DATA_YAML = DS / 'data.yaml'
DATA_YAML.write_text(yaml.safe_dump({'path': str(DS), 'train': str(DS / 'train' / 'images'), 'val': str(DS / 'val' / 'images'),
                                     'test': str(DS / 'test_own' / 'images'), 'nc': 1, 'names': ['ball']}))
# dimensione della palla nei tuoi frame di test
sz = [float(l.split()[3]) * IMG_W for p in (DS / 'test_own' / 'labels').glob('*.txt') for l in p.read_text().splitlines() if l.strip()]
if sz: print(f'palla nei tuoi frame di test: larghezza mediana {np.median(sz):.1f} px (5°–95° perc.: {np.percentile(sz, 5):.1f}–{np.percentile(sz, 95):.1f} px)')

## 4. Addestramento del detector della palla (riprende da solo se Colab si disconnette)

In [ ]:
import torch
RUNS = E4 / 'runs'; NAME = f'ball_{Path(BASE_MODEL).stem}_{TRAIN_IMGSZ}'
last, best = RUNS / NAME / 'weights' / 'last.pt', RUNS / NAME / 'weights' / 'best.pt'
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
batch = 16 if ('L4' in gpu or 'A100' in gpu) else 8
done = (RUNS / NAME / 'results.csv').is_file() and best.is_file() and len(pd.read_csv(RUNS / NAME / 'results.csv')) >= EPOCHS
if done:
    print('[skip] addestramento già completato:', best)
elif last.is_file():
    print('Riprendo da', last); YOLO(str(last)).train(resume=True)
else:
    YOLO(BASE_MODEL).train(data=str(DATA_YAML), imgsz=TRAIN_IMGSZ, epochs=EPOCHS, patience=PATIENCE, batch=batch, seed=SEED,
                           project=str(RUNS), name=NAME, exist_ok=True, single_cls=True,
                           mosaic=1.0, close_mosaic=10, scale=0.5, fliplr=0.5, degrees=0.0, hsv_v=0.4, workers=4, plots=True)
shutil.copy(best, ROOT / 'models' / 'ball_detector.pt')
print('Modello salvato in models/ball_detector.pt | GPU:', gpu, '| batch:', batch)

## 5. Confronto sui tuoi frame di test (mai visti in addestramento)

Tre detector: **vecchio a 640** (come in E1), **vecchio a 1280** (come nel notebook 15), **nuovo a 1280**. Metriche:
- **AP50** (IoU ≥ 0,5) e **AP50:95**, come in E1;
- **precisione / richiamo / F1** alla soglia operativa 0,25;
- **richiamo e precisione "sul centro"**: un rilevamento è corretto se il suo centro cade entro 1,5 diametri dalla palla vera. Per il tracking della palla conta trovarla nel posto giusto più che disegnare una box perfetta su un oggetto di 10 px, dove l'IoU è molto severo.

In [ ]:
def load_gt(split):
    out = {}
    for img in sorted((DS / split / 'images').glob('*.jpg')):
        h, w = cv2.imread(str(img)).shape[:2]; b = []
        for l in (DS / split / 'labels' / (img.stem + '.txt')).read_text().splitlines():
            if l.strip():
                _, cx, cy, bw, bh = map(float, l.split()[:5]); b.append([(cx - bw / 2) * w, (cy - bh / 2) * h, (cx + bw / 2) * w, (cy + bh / 2) * h])
        out[img] = np.array(b).reshape(-1, 4)
    return out

def predict(model, imgs, imgsz, classes):
    out, t = {}, []
    for p in tqdm(imgs, leave=False):
        t0 = time.perf_counter(); r = model.predict(str(p), imgsz=imgsz, conf=0.001, classes=classes, verbose=False, max_det=50)[0]; t.append(time.perf_counter() - t0)
        out[p] = np.column_stack([r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy()]).reshape(-1, 5)
    return out, 1 / np.mean(t[3:] if len(t) > 5 else t)

def iou(a, b):
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None); ar = lambda r: (r[:, 2] - r[:, 0]) * (r[:, 3] - r[:, 1])
    return inter / (ar(a)[:, None] + ar(b)[None, :] - inter + 1e-9)

def center_ok(a, b):   # centro del rilevamento entro 1,5 diametri dal centro della palla vera
    ca = (a[:, None, :2] + a[:, None, 2:4]) / 2; cb = (b[None, :, :2] + b[None, :, 2:4]) / 2
    diam = np.maximum(b[None, :, 2] - b[None, :, 0], b[None, :, 3] - b[None, :, 1])
    return np.where(np.linalg.norm(ca - cb, axis=2) <= 1.5 * np.maximum(diam, 4), 1.0, 0.0)

def evaluate(preds, gts, sim, thr):
    recs, n_gt = [], sum(len(g) for g in gts.values())
    for p, g in gts.items():
        P = preds[p]; P = P[np.argsort(-P[:, 4])]; used = np.zeros(len(g), bool)
        S = sim(P[:, :4], g) if len(P) and len(g) else np.zeros((len(P), len(g)))
        for k in range(len(P)):
            j = int(np.argmax(np.where(used, -1, S[k]))) if len(g) else -1
            ok = j >= 0 and not used[j] and S[k, j] >= thr
            if ok: used[j] = True
            recs.append((P[k, 4], ok))
    recs.sort(key=lambda r: -r[0]); tp = np.cumsum([r[1] for r in recs]); fp = np.cumsum([not r[1] for r in recs])
    rec = tp / max(n_gt, 1); prec = tp / np.maximum(tp + fp, 1)
    mrec = np.concatenate([[0], rec, [1]]); mpre = np.concatenate([[1], prec, [0]])
    for i in range(len(mpre) - 2, -1, -1): mpre[i] = max(mpre[i], mpre[i + 1])
    ap = float(np.sum((mrec[1:] - mrec[:-1]) * mpre[1:]))
    confs = np.array([r[0] for r in recs]); k = (confs >= 0.25).sum()
    P25 = prec[k - 1] if k else np.nan; R25 = rec[k - 1] if k else 0.0
    return ap, P25, R25

new = YOLO(str(ROOT / 'models' / 'ball_detector.pt'))
DETECTORS = {'vecchio_640': (old, 640, OLD_BALL), 'vecchio_1280': (old, 1280, OLD_BALL), 'nuovo_1280': (new, TRAIN_IMGSZ, None)}
rows = []
for split in ['test_own', 'test_v18']:
    gts = load_gt(split)
    for name, (m, sz, cls) in DETECTORS.items():
        preds, fps = predict(m, list(gts), sz, cls)
        ap50, p, r = evaluate(preds, gts, iou, 0.5)
        ap5095 = np.mean([evaluate(preds, gts, iou, t)[0] for t in np.arange(.5, .96, .05)])
        apc, pc, rc = evaluate(preds, gts, center_ok, 0.5)
        rows.append({'test': split, 'detector': name, 'AP50': ap50, 'AP50:95': ap5095, 'P@0.25': p, 'R@0.25': r,
                     'F1@0.25': 2 * p * r / (p + r) if p + r else 0, 'AP_centro': apc, 'P_centro@0.25': pc, 'R_centro@0.25': rc, 'FPS': fps})
res = pd.DataFrame(rows).round(3); res.to_csv(E4 / 'confronto_palla.csv', index=False); display(res)

## 6. Prova su una clip intera: in quanti frame si trova la palla?

In [ ]:
RATE_CLIP = CLIPS[0]; RATE_MAX_S = 60; RATE_STEP = 2       # primi 60 s, un frame ogni 2
cap = cv2.VideoCapture(str(RATE_CLIP)); fps = cap.get(cv2.CAP_PROP_FPS) or 30
frames = []
for fi in range(int(RATE_MAX_S * fps)):
    ok, f = cap.read()
    if not ok: break
    if fi % RATE_STEP == 0: frames.append(cv2.resize(f, (IMG_W, int(round(f.shape[0] * IMG_W / f.shape[1])))) if f.shape[1] != IMG_W else f)
cap.release()
rate = []
for name, (m, sz, cls) in DETECTORS.items():
    found = [len(m.predict(f, imgsz=sz, conf=0.25, classes=cls, verbose=False)[0].boxes) > 0 for f in tqdm(frames, desc=name, leave=False)]
    rate.append({'detector': name, 'frame_con_palla_%': round(100 * np.mean(found), 1)})
rate = pd.DataFrame(rate); rate.to_csv(E4 / 'frequenza_palla_clip.csv', index=False)
print(f'{RATE_CLIP.name}: {len(frames)} frame analizzati'); display(rate)
print('Nota: la palla non è visibile in tutti i frame (mani, corpi, fuori campo): il 100% non è l\'obiettivo; conta il confronto e la precisione della sezione 5.')

## 7. Figure per la tesi

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
t = res[res.test == 'test_own'].set_index('detector')
t[['AP50', 'AP_centro', 'R_centro@0.25']].plot.bar(ax=axes[0], rot=0, color=['#2a6f97', '#61a5c2', '#e07a5f'])
axes[0].set_ylim(0, 1); axes[0].set_title('Palla — tuoi frame NBA di test'); axes[0].grid(axis='y', alpha=.3)
rate.set_index('detector')['frame_con_palla_%'].plot.bar(ax=axes[1], rot=0, color='#81b29a')
axes[1].set_title(f'Frame con palla (conf ≥ 0,25), {RATE_CLIP.stem}'); axes[1].set_ylabel('%'); axes[1].grid(axis='y', alpha=.3)
for ax in axes: ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout(); fig.savefig(E4 / 'confronto_palla.png', dpi=170); plt.show()

# esempi qualitativi: 6 frame di test con GT (verde), vecchio_1280 (rosso), nuovo (giallo)
gts = load_gt('test_own'); ex = random.Random(1).sample(list(gts), min(6, len(gts)))
fig, axes = plt.subplots(2, 3, figsize=(18, 7))
for ax, p in zip(axes.ravel(), ex):
    img = cv2.imread(str(p))
    for b in gts[p]: cv2.rectangle(img, tuple(map(int, b[:2])), tuple(map(int, b[2:4])), (0, 200, 0), 2)
    for (m, sz, cls), c in [(DETECTORS['vecchio_1280'], (0, 0, 255)), (DETECTORS['nuovo_1280'], (0, 220, 255))]:
        for b in m.predict(img, imgsz=sz, conf=0.25, classes=cls, verbose=False)[0].boxes.xyxy.cpu().numpy():
            cv2.circle(img, (int((b[0] + b[2]) / 2), int((b[1] + b[3]) / 2)), 14, c, 2)
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); ax.axis('off')
fig.suptitle('Verde = palla vera · rosso = detector vecchio (1280) · giallo = detector nuovo'); fig.tight_layout()
fig.savefig(E4 / 'esempi_palla.png', dpi=130); plt.show()
print('Risultati in', E4)